# Phase 2: Model Validation & Stability Analysis

This notebook empirically evaluates the hyperparameter choices of the HMM, testing the stability of the model using rigorous out-of-sample principles.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

from src.data.loader import load_config, download_prices
from src.features.engineering import build_features, FEATURE_COLUMNS


## 1. Data Setup (No Look-Ahead)
Using the full dataset to tune hyperparameters leaks future macro regimes into the model design. We strictly isolate an initial training window to perform model selection. 

* **Feature Dimension ($d=8$):** We reduced our feature space down to exactly 8 orthogonal indicators across momentum, trend, volatility, and volume.
* **Initialization Window:** We expanded our initialization window from 4 years to **9 years** (2000 through 2008). The original 4-year window lacked the macro diversity to anchor 4 states. By including 2008, the model explicitly learns the mathematical signature of a Great Financial Crisis before it begins walk-forward predictions out-of-sample.


In [ ]:
cfg = load_config('../config/config.yaml')
data = download_prices(config=cfg, cache_dir='../data/raw')
spx = data['spx']
features = build_features(spx, spx['VIX_Close'])

train_df = features[features.index.year <= 2008]
X_train_raw = train_df[FEATURE_COLUMNS].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)

N, d = X_train.shape
print(f"Initial training window: N={N} observations, d={d} features.")

## 2. Diagonal Covariance vs Parameter Count

By switching `covariance_type` from `full` to `diag`, we force the model to look at orthogonal feature shifts and massively restrict the degrees of freedom.

The number of free parameters $p$ for a diagonal covariance matrix is:
$p = (k - 1) + k(k - 1) + k \cdot d + k \cdot d$

With $d=8$ features and $k=4$ states, we have exactly **79 parameters** against **2,238 observations** (an incredibly robust ratio of ~28 observations per parameter).


## 3. Seed Stability

We refit the $k=4$ diagonal model across 20 different random seeds and measure the Adjusted Rand Index (ARI) against the baseline run. By feeding the model a diverse 9-year cycle, we expect structural convergence.


In [ ]:
from sklearn.metrics import adjusted_rand_score
from hmmlearn.hmm import GaussianHMM

# Baseline model
baseline = GaussianHMM(n_components=4, covariance_type='diag', n_iter=2000, random_state=42, tol=1e-4)
baseline.fit(X_train)
baseline_states = baseline.predict(X_train)

aris = []
for seed in range(20):
    m = GaussianHMM(n_components=4, covariance_type='diag', n_iter=2000, random_state=seed, tol=1e-4)
    m.fit(X_train)
    states = m.predict(X_train)
    aris.append(adjusted_rand_score(baseline_states, states))

print(f"Mean ARI: {np.mean(aris):.4f}")
print(f"Min ARI:  {np.min(aris):.4f}")
print(f"Max ARI:  {np.max(aris):.4f}")

## Conclusion

By applying three critical constraints:
1. Slashed feature space from 18 to 8
2. Switched to diagonal covariance matrices
3. Provided sufficient training window (2000-2008)

We observe that the `GaussianHMM` successfully converges to stable states, evidenced by high Adjusted Rand Scores across different random seeds.